Last edited: 1/28/2026, 3:38 PM CST
# Benchmarking MLE algorithm

In [1]:
# import base packages for data analysis
import pandas as pd
import xarray as xr

from evt_heat_waves.config import DATA_ROOT
from evt_heat_waves.utils import extract_model_name
from evt_heat_waves.mip_fit.cmip_dataclass import CMIP6EnsembleConfig

## Global attributes

In [ ]:
fit = 'nonstat_gumbel_only_loc_trend'
anom_types = ['raw', 'trend', 'annmean']
TMIN = 1979

## ERA5

In [3]:
era5_variable = 't2m_annual_max'
era5_grid = '1deg'

era5_perf_stats = {}
era5_perf_stats['MLE_success_rate'] = {}

In [4]:
for anom_type in anom_types:
    tmp_ds = xr.open_dataset(
        DATA_ROOT / 'ERA5'/ 'gev' / f'era5_{era5_variable}_{era5_grid}_landonly_gev_{fit}_TMIN{TMIN}_{anom_type}.nc'
    )

    era5_perf_stats['MLE_success_rate'][anom_type] = tmp_ds.MLE_success_rate
    
    tmp_ds.close()

In [5]:
era5_df = pd.DataFrame(era5_perf_stats)
era5_df.to_csv(
    DATA_ROOT / 'stats' / f'mle_perf_era5_{fit}.csv'
)

## CMIP6

In [6]:
cmip_variable = 'tas_annual_max'

CMIPConfig = CMIP6EnsembleConfig.from_yaml(
    '/project/bbcael/ambauer/gev-heat-waves/config/meta.yaml',
    '/project/bbcael/ambauer/gev-heat-waves/config/qc.yaml'
)

cmip_perf_stats = {}
cmip_perf_stats['MLE_success_rate'] = {}

In [7]:
for anom_type in anom_types:
    # tmp empty list
    tmp_success_rates = []

    # make file/model matcher for 
    data_path = DATA_ROOT / 'CMIP6' / cmip_variable / 'gev'

    # Make all landonly file names
    fnames = [f for f in data_path.glob(f"*{fit}*{anom_type}*.nc") if "allmems" not in f.name]  # screen out allmems results

    modelname_filepath_matcher = {
    extract_model_name(f): f for f in fnames
    }

    for m in CMIPConfig.iter_active_models(cmip_variable):
        fname = modelname_filepath_matcher[m.name]
        tmp_ds = xr.open_dataset(fname)

        tmp_success_rates.append(tmp_ds.MLE_success_rate)

        tmp_ds.close()

    cmip_perf_stats['MLE_success_rate'][anom_type] = tmp_success_rates

In [8]:
cmip_perf_stats['MLE_success_rate']

{'raw': [1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0],
 'trend': [1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0],
 'annmean': [1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0]}

In [9]:
cmip_df = pd.DataFrame.from_dict(cmip_perf_stats['MLE_success_rate'], orient='index',
                                 columns=[m.name for m in CMIPConfig.iter_active_models(cmip_variable)])
cmip_df

,AWI-CM-1-1-MR,BCC-CSM2-MR,CAMS-CSM1-0,CESM2-WACCM,CMCC-CM2-SR5,CMCC-ESM2,CNRM-CM6-1-HR,CNRM-ESM2-1,EC-Earth3-CC,EC-Earth3-Veg,...,KACE-1-0-G,KIOST-ESM,MIROC-ES2L,MIROC6,MPI-ESM1-2-HR,MPI-ESM1-2-LR,NorCPM1,NorESM2-LM,TaiESM1,UKESM1-0-LL
raw,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
trend,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
annmean,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


In [10]:
cmip_df.mean(axis=1)

raw        1.0
trend      1.0
annmean    1.0
dtype: float64

In [11]:
cmip_df.to_csv(
    DATA_ROOT / 'stats' / f'mle_perf_cmip_{fit}.csv'
)